# Generate Test Data
Generates data in tests/test_data/




In [ ]:

import sys
sys.path.insert(0, "/mnt/c/Users/hans/OneDrive/Python3/packages/cdxcore") # do this before importing anything else


import os
os.environ["QUANTDATA_TEST_MODE"] = "generate_tests" # do this before importing anything else

%load_ext autoreload
%autoreload 1

%aimport quantdata
%aimport quantdata.biz_calendar
%aimport quantdata.utils.arbfree
%aimport quantdata.widb
%aimport quantdata.widb.client
%aimport quantdata.widb.day
%aimport quantdata.widb.arbfree

from quantdata import env# triggers construction of caches.
from quantdata.biz_calendar import next_business_day
from cdxcore.verbose import Context

def generate_day(ticker, d):
    from quantdata.widb.day import collect_raw_day, enrich_day, ExpiryFilter, RawDayFilter
    from quantdata.widb.arbfree import mk_arb_free_day, MkArbFree

    raw_day_filter = RawDayFilter()
    expiry_filter = ExpiryFilter()
    mk_arb_free = MkArbFree()
    verbose = Context.all

    raw_day = collect_raw_day(ticker, d, raw_day_filter=raw_day_filter, verbose=verbose, raise_if_bad=False)
    enr_day = enrich_day(ticker, d, raw_day_filter=raw_day_filter, expiry_filter=expiry_filter, verbose=verbose, raise_if_bad=False, override_cache_mode="off")

    buggy   = enr_day[enr_day['pure_ask_price'] <= enr_day['pure_bid_price']]
    assert buggy.empty, f"Found {len(buggy)} rows with ask price <= bid price for {ticker} {d}:\n{buggy[['expiry','strike','spot','pure_ask_price', 'pure_bid_price','call_ask','call_bid','mkt_call_ask', 'mkt_call_bid','mkt_put_ask', 'mkt_put_bid']]}"


    af_day = mk_arb_free_day(ticker, d, mk_arb_free=mk_arb_free, raw_day_filter=raw_day_filter, expiry_filter=expiry_filter, verbose=verbose, raise_if_bad=False)

tickers = ["SPX", "AAPL"]
dates = ["2006-01-31", "2005-11-18", "2024-03-03", "2025-01-02", "2024-12-24", "2010-07-05", "2011-07-06"]
dates = [next_business_day(d,count_start=True) for d in dates]

for ticker in tickers:
    for d in dates:
        print(f"Generating data for {ticker} {d}")
        generate_day(ticker, d)
        break


/mnt/c/Users/hans/OneDrive/Python3/quant_data/quantdata/__init__.py:272: UserWarning: Initializing quantdata. **WARNING: quantdata is in test data generation mode **.
Caching directories are set as follows:
 raw: /mnt/c/Users/hans/OneDrive/Python3/quant_data/tests/stored_data/raw/
 processing: /mnt/c/Users/hans/OneDrive/Python3/quant_data/tests/stored_data/processing/
 training: /mnt/c/Users/hans/OneDrive/Python3/quant_data/tests/stored_data/training/
 experiments: /mnt/c/Users/hans/OneDrive/Python3/quant_data/tests/stored_data/experiments/
 temp: /tmp/tmpbxo6h9z0/quantdata_test_temp/
Default cache mode is set to 'on'.


Generating data for SPX 2006-01-31
00: Collecting raw market data for SPX on 2006-01-31.
00: SPX 2006-01-31: collected 300/641 options. Removed: 2/641 'bad' options and 19/639 options with bid <= 0.011.
01:   Computing implied vols for 53 options:
02:     Used intrinsic value for 0 options.
02:     Used maximum volatility 5.000 for 0 options.
02:     Used minimum volatility 0.010 for 14 options.
01:   Implied vol calculation for 53 options finished using 4/100 iterations. Converged for all options. Max error 0.000, L1 error 0.000, L2 error 0.000. This took 3.45ms. 
01:   Computing implied vols for 53 options:
02:     Used intrinsic value for 0 options.
02:     Used maximum volatility 5.000 for 0 options.
02:     Used minimum volatility 0.010 for 15 options.
01:   Implied vol calculation for 53 options finished using 4/100 iterations. Converged for all options. Max error 0.001, L1 error 0.000, L2 error 0.000. This took 3.04ms. 
Ask
Spread [3.91999257e-05 7.83998543e-05 7.83998543e-05 7.

AssertionError: Internal error: SPX 2006-01-31: found non-positive bid-ask spread 2006-02-18am. Please contact the authors with an example.